In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_musa
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import math
from tqdm import tqdm

Error in cpuinfo: prctl(PR_SVE_GET_VL) failed


In [2]:
file_name = 'data/data_train/training_data.dat'
data = pd.read_csv(file_name)

train, val = train_test_split(data, test_size=0.2, random_state=42)

input_columns = [1, 2, 3]
initial_c = train.iloc[:, input_columns].values
output_columns = 5
t12_all = train.iloc[:, output_columns].values

input_scaler = StandardScaler()
output_scaler = StandardScaler()

input_normalized = input_scaler.fit_transform(initial_c)
output_normalized = output_scaler.fit_transform(t12_all.reshape(-1, 1)).reshape(-1)

class CustomDataset(Dataset):
    def __init__(self, initial_c, t12):
        self.input = torch.tensor(initial_c, dtype=torch.float32)
        self.label = torch.tensor(t12, dtype=torch.float32)

    def __len__(self):
        return len(self.label);

    def __getitem__(self, i):
        return self.input[i], self.label[i]

dataset = CustomDataset(input_normalized, output_normalized)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=4, persistent_workers=True)

## 训练阶段 training phase

In [3]:
class BaselineModel(nn.Module):
    def __init__(self):
        super(BaselineModel, self).__init__()
        self.fc1 = nn.Linear(3, 12)
        self.fc2 = nn.Linear(12, 24)
        self.fc3 = nn.Linear(24, 36)
        self.fc4 = nn.Linear(36, 1)
        self.ac = nn.LeakyReLU()
    
    def forward(self, x):
        x1 = self.ac(self.fc1(x))
        x2 = self.ac(self.fc2(x1))
        x3 = self.ac(self.fc3(x2))
        x4 = self.fc4(x3)

        return x4

In [4]:
num_epochs = 10000
device = torch.device("musa" if torch.musa.is_available() else "cpu")
model = BaselineModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.train()
for epoch in tqdm(range(num_epochs)):
    tot_loss = 0
    batch_num = 0
    for input, label in dataloader:
        #print(input)
        #print(label)
        input = input.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        output = model(input)
        output = torch.squeeze(output)
        loss = criterion(output, label)
        tot_loss += loss
        batch_num += 1
        loss.backward()
        optimizer.step()

    avg_loss = tot_loss / batch_num

100%|██████████████████████████████████████████| 10000/10000 [05:45<00:00, 28.94it/s]


## 验证阶段 validation phase

In [5]:
input_columns = [1, 2, 3]
initial_c = input_scaler.transform(val.iloc[:, input_columns].values)
output_columns = 5
t12_all = val.iloc[:, output_columns].values

In [6]:
num = t12_all.size
input = torch.tensor(initial_c, dtype=torch.float32, device=device)
t12_act = torch.tensor(t12_all, dtype=torch.float32, device=device)

In [7]:
model.eval()
tot_score = 0

with torch.no_grad():
    for i in range(num):
        output = output_scaler.inverse_transform(model(input[i]).unsqueeze(1).cpu()).squeeze()
        pred = output.item()
        act = t12_act[i].item()
        score = max(0, 1 - math.log(1+0.1*abs(pred-act))/5)
        tot_score += score
        
        print(f't1/2 predicted: {pred: .4f}; t1/2 actual: {act: .4f}; score: {score}')

avg_score = tot_score / num
print(f'final score: {avg_score}')

t1/2 predicted:  138.9610; t1/2 actual:  152.3800; score: 0.8298076534872676
t1/2 predicted:  606.9339; t1/2 actual:  606.2100; score: 0.9860231360921142
t1/2 predicted:  27.3514; t1/2 actual:  19.4200; score: 0.8832065951377702
t1/2 predicted:  25.8879; t1/2 actual:  18.1700; score: 0.8856017239367214
t1/2 predicted:  236.2671; t1/2 actual:  235.1800; score: 0.9793605880666432
t1/2 predicted:  420.1291; t1/2 actual:  418.3700; score: 0.9675911671477411
t1/2 predicted:  739.8535; t1/2 actual:  742.8000; score: 0.948352801961088
t1/2 predicted:  595.1683; t1/2 actual:  587.9900; score: 0.8917879225506076
t1/2 predicted:  481.8013; t1/2 actual:  490.8200; score: 0.8714323335049088
t1/2 predicted:  1453.6319; t1/2 actual:  1457.9000; score: 0.9289115559194385
t1/2 predicted:  134.1864; t1/2 actual:  151.3600; score: 0.8000674895170664
t1/2 predicted:  1178.6836; t1/2 actual:  1173.3000; score: 0.9138578035643968
t1/2 predicted:  594.1520; t1/2 actual:  583.0900; score: 0.8510229726487237


## 测试阶段 testing phase

### 验证集 validation set
参赛者可在提交平台看到public score

In [8]:
data_file_name = 'data/data_val/val_data_question.dat'
data = pd.read_csv(data_file_name)
input_columns = [1, 2, 3]
initial_c = input_scaler.transform(data.iloc[:, input_columns].values)
input_ = torch.tensor(initial_c, dtype=torch.float32, device=device)


model.eval()
tot_score = 0
pd_pred = pd.DataFrame(columns = ['Exp #', 't12_simulated'])

with torch.no_grad():
    for i in range(len(initial_c)):
        t12_pred = output_scaler.inverse_transform(model(input_[i]).unsqueeze(1).cpu()).squeeze()
        pred = t12_pred.item()
        pd_pred.loc[i, 'Exp #']= i
        pd_pred.loc[i, 't12_simulated'] = pred


pd_pred['t12_simulated'] = pd_pred['t12_simulated'].apply(lambda x: f"{x:.4e}")
pd_pred.to_csv('submission_val.csv', index=False)

### 测试集 test set
参赛者无法获取测试集或在提交后得到测试集得分
The testing sets and its results are made not accessible for contestants.

In [9]:
data_file_name = 'data/data_test/test_data_question.dat'
data = pd.read_csv(data_file_name)
input_columns = [1, 2, 3]
initial_c = input_scaler.transform(data.iloc[:, input_columns].values)
input_test = torch.tensor(initial_c, dtype=torch.float32, device=device)


model.eval()
pd_pred_test = pd.DataFrame(columns = ['Exp #', 't12_simulated'])

with torch.no_grad():
    for i in range(len(initial_c)):
        t12_pred = output_scaler.inverse_transform(model(input_test[i]).unsqueeze(1).cpu()).squeeze()
        pred = t12_pred.item()
        pd_pred_test.loc[i, 'Exp #']= i
        pd_pred_test.loc[i, 't12_simulated'] = pred




pd_pred_test['t12_simulated'] = pd_pred_test['t12_simulated'].apply(lambda x: f"{x:.4e}")
pd_pred_test.to_csv('submission_test.csv', index=False)